In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time
from openai import OpenAI
from google.colab import userdata

In [ ]:
# FIX (Bug 2): Use the Instruct model, not the base model.
# The base model ("Meta-Llama-3.1-8B") won't reliably follow instructions
# or produce structured output. The Instruct variant was fine-tuned for that.
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None          # Auto-detect: float16 for T4/V100, bfloat16 for Ampere+
load_in_4bit = True   # 4-bit quantization to fit in GPU memory

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",  # <-- Instruct model
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not find `steps_per_generation` in grpo_trainer
Unsloth: Could not find `generation_batch_size` in grpo_trainer
==((====))==  Unsloth 2026.3.8: Fast Llama patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-bnb-4bit as a legacy tokenizer.


In [ ]:
# FIX (Bug 2 continued): LoRA / PEFT setup REMOVED.
# get_peft_model() adds trainable adapter weights — that's for fine-tuning,
# not inference. We're using the Instruct model as-is, so we skip this entirely.

# Instead, put the model into fast inference mode:
FastLanguageModel.for_inference(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm):

In [ ]:
# The alpaca_prompt template and formatting_prompts_func from the Unsloth
# fine-tuning tutorial are not needed for inference-only judging.
# (Removed to avoid confusion — we build our evaluation prompt directly in cell 7.)

In [ ]:
df=pd.read_csv("results_fetch_econ_full.csv")
print(f"Total rows in CSV: {len(df)}")

Total rows in CSV: 30


In [ ]:
df.head()

,title,abstract,license,url,html_url,date
0,(Ab)using Indifference: Purification in Commun...,Recent papers in communication games construct...,http://creativecommons.org/licenses/by-nc-sa/4.0/,https://arxiv.org/abs/2602.23098,https://arxiv.org/html/2602.23098,2026-02-26
1,Forecasting on the Accuracy-Timeliness Frontie...,We re-examine the traditional Mean-Squared Err...,http://creativecommons.org/licenses/by/4.0/,https://arxiv.org/abs/2602.23087,https://arxiv.org/html/2602.23087,2026-02-26
2,Randomization Tests in Switchback Experiments,Switchback experiments--alternating treatment ...,http://creativecommons.org/licenses/by/4.0/,https://arxiv.org/abs/2602.23257,https://arxiv.org/html/2602.23257,2026-02-26
3,The Inference Bottleneck: Antitrust and Neutra...,"As generative AI commercializes, competitive a...",http://creativecommons.org/licenses/by/4.0/,https://arxiv.org/abs/2602.22750,https://arxiv.org/html/2602.22750,2026-02-26
4,Modelling the Index of Sustainable Economic We...,Given the challenge of achieving societal welf...,http://creativecommons.org/licenses/by/4.0/,https://arxiv.org/abs/2602.21971,https://arxiv.org/html/2602.21971,2026-02-25


In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time

def fetch_html_body_content(html_url):

    try:
        r = requests.get(html_url, timeout=30)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
    except Exception as e:
        print("⚠ HTML 请求失败:", e)
        return None, "html_fetch_failed"

    body = soup.find("body")
    if not body:
        return None, "no_body"

    lines = [line.strip() for line in body.stripped_strings if line.strip()]
    text = "\n".join(lines)

    if not text:
        return None, "empty_text"

    # 移除 Abstract
    text = re.sub(
        r'Abstract[:\s].*?(?=(Introduction|1\.\sIntroduction))',
        '',
        text,
        flags=re.S|re.I
    )

    # 截断 Acknowledgment
    ack_patterns = [
        r'Acknowledgment',
        r'Acknowledgements',
        r'ACKNOWLEDGMENT',
        r'ACKNOWLEDGEMENTS'
    ]

    for pat in ack_patterns:
        match = re.search(pat, text, re.I)
        if match:
            text = text[:match.start()]
            break

    text = "\n".join(line for line in text.splitlines() if line.strip())

    return text, "html_success"

In [ ]:
import traceback
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
# from pipeline import fetch_html_body_content

# Initialize columns for the results
df["llama_prompt"] = ""  # Ensure the target column exists

# --- DATA QUALITY FIX: strip arXiv HTML navigation boilerplate ---
BOILERPLATE_EXACT = {
    "Report GitHub Issue", "Submit without GitHub", "Submit in GitHub",
    "Back to arXiv", "Why HTML?", "Report Issue", "Content selection saved",
    "Describe the issue below", "×",
}

BOILERPLATE_SUBSTRINGS = [
    "Report GitHub Issue",
    "Back to arXiv",
    "Why HTML?",
    "Report Issue",
    "Submit without GitHub",
    "Submit in GitHub",
    "Back to Introduction",
    "Back to ",
    "Content selection saved",
    "Describe the issue below",
]

def get_article_snippet(full_text, max_chars=4000):
    """
    Strip the arXiv HTML preamble and return the first max_chars of actual prose.
    """
    lines = full_text.splitlines()

    # Phase 1: Identify where the actual article body starts
    body_start = 0
    for i, line in enumerate(lines):
        stripped = line.strip()
        if not stripped:
            continue
        # Look for a line that resembles prose (long, contains spaces and lowercase letters)
        if len(stripped) > 80 and " " in stripped and re.search(r"[a-z]", stripped):
            body_start = i
            break

    # Phase 2: Filter out remaining boilerplate and section numbers
    cleaned_lines = []
    for line in lines[body_start:]:
        stripped = line.strip()
        if not stripped:
            if cleaned_lines:
                cleaned_lines.append("")
            continue
        if any(bp in stripped for bp in BOILERPLATE_SUBSTRINGS):
            continue
        if stripped in BOILERPLATE_EXACT:
            continue
        if re.match(r"^[A-Z]?\.?\d+(\.\d+)*$", stripped):
            continue
        cleaned_lines.append(line)

    cleaned = "\n".join(cleaned_lines)
    return cleaned[:max_chars]


# --- Main evaluation loop ---
for index, row in df.iterrows():
    try:
        html_url = row.get("html_url", None)
        if not html_url:
            continue

        # Fetch the content (assuming fetch_html_body_content is defined elsewhere)
        article, status = fetch_html_body_content(html_url)
        if status != "html_success":
            print(f"Row {index}: HTML fetch failed ({status})")
            continue

        article_snippet = get_article_snippet(article)

        # Build chat message structure
        messages = [
            {"role": "user", "content": f"""
You are a professional science popularization writer.

Task:
Rewrite the following scientific paper content into a clear and accessible explanation.

Requirements:
- Audience: general public with high-school background
- Length: 150–200 words
- Focus on the main idea, method, and contribution
- Avoid equations and heavy jargon

Paper content:
{article_snippet}
"""}
        ]

        encoded = tokenizer.apply_chat_template(
            messages,
            return_tensors="pt",
            add_generation_prompt=True,
            truncation=True,
        )

        # move to device safely
        encoded = {k: v.to(device) for k, v in encoded.items()}

        outputs = model.generate(
            input_ids=encoded["input_ids"],
            attention_mask=encoded.get("attention_mask"),
            max_new_tokens=2000,
        )

        # ✅ 正确截取生成部分
        input_length = encoded["input_ids"].shape[-1]
        generated_tokens = outputs[:, input_length:]

        output_text = tokenizer.decode(
            generated_tokens[0],
            skip_special_tokens=True
        ).strip()

        # Update the specific row's 'llama_prompt' column with the transferred text
        # Using .at[index, col] ensures we only modify the current row
        df.at[index, "llama_prompt"] = output_text

        print(f"--- Row {index}: Successfully processed ---")

    except Exception as e:
        # Catch and report errors to prevent the loop from breaking
        print(f"Error processing Row {index}: {e}")
        traceback.print_exc()
        continue

# Save the final results to CSV
df.to_csv("results_llama_econ.csv", index=False)
print(f"\nSaved {len(df)} rows to results_llama_phy.csv")

--- Row 0: Successfully processed ---
--- Row 1: Successfully processed ---
--- Row 2: Successfully processed ---
--- Row 3: Successfully processed ---
⚠ HTML 请求失败: 404 Client Error: Not Found for url: https://arxiv.org/html/2602.21971
Row 4: HTML fetch failed (html_fetch_failed)
--- Row 5: Successfully processed ---
⚠ HTML 请求失败: 404 Client Error: Not Found for url: https://arxiv.org/html/2602.21843
Row 6: HTML fetch failed (html_fetch_failed)
--- Row 7: Successfully processed ---
⚠ HTML 请求失败: 404 Client Error: Not Found for url: https://arxiv.org/html/2602.21495
Row 8: HTML fetch failed (html_fetch_failed)
--- Row 9: Successfully processed ---
⚠ HTML 请求失败: 404 Client Error: Not Found for url: https://arxiv.org/html/2602.20378
Row 10: HTML fetch failed (html_fetch_failed)
⚠ HTML 请求失败: 404 Client Error: Not Found for url: https://arxiv.org/html/2602.20356
Row 11: HTML fetch failed (html_fetch_failed)
⚠ HTML 请求失败: 404 Client Error: Not Found for url: https://arxiv.org/html/2602.20327
Row